# 01 - Explore the raw data

Start here. This notebook answers *what is actually in these files* before any modelling:
how sparse demand is, how much of it is stock-out censored, what the seasonality looks
like, and which SKUs are forecastable at all.

Assumes `demandml ingest` and `demandml panel` have been run.

In [ ]:
import polars as pl
import matplotlib.pyplot as plt
import seaborn as sns
from demandml import config as C
from demandml.panel import load_panel
from demandml.profiling import column_report, demand_classes, panel_report

pl.Config(tbl_rows=30, tbl_cols=15, fmt_str_lengths=40)
sns.set_theme(style="whitegrid", palette="deep")
plt.rcParams["figure.figsize"] = (12, 4)

panel = load_panel("daily")
print(f"{panel.select(pl.len()).collect().item():,} rows")

## Headline data quality

The numbers that decide whether any score you get later is real.

In [ ]:
report = panel_report('daily')

## Per-column nulls and ranges

Look for columns that are ~100% null (drop them), columns that are 100% zero (drop
them), and columns whose null rate changes over time (those are features that only
exist for recent years -- they will quietly stop working on older folds).

In [ ]:
cr = column_report(panel, sample=400_000)
cr.filter(pl.col("null_pct") > 0).head(30)

In [ ]:
# Columns that are entirely constant carry no information.
cr.filter(pl.col("n_unique") <= 1)

## How sparse is demand?

A large share of SKU-days have zero sales (the next cell prints it for your data).
That single fact rules out MAPE,
makes squared-error loss a poor choice, and makes Croston-style methods genuinely
competitive.

In [ ]:
units = panel.select(C.TARGET).collect().get_column(C.TARGET)
print(f"zero days: {(units == 0).mean():.1%}")
print(f"mean {units.mean():.2f} | median {units.median():.0f} | p99 {units.quantile(0.99):.0f} | max {units.max():.0f}")

fig, ax = plt.subplots(1, 2, figsize=(13, 4))
sns.histplot(units.filter(units <= 50).to_numpy(), bins=51, ax=ax[0])
ax[0].set(title="daily units (<=50)", xlabel="units")
sns.histplot(units.filter(units > 0).log1p().to_numpy(), bins=60, ax=ax[1])
ax[1].set(title="log1p(units) | units > 0", xlabel="log1p units")
plt.tight_layout()

## Demand classes

The Syntetos-Boylan quadrants. These four groups want different models, and a single
average WAPE hides that completely -- always break your results down by this.

In [ ]:
cls = demand_classes(panel)
summary = (cls.group_by("demand_class")
             .agg(pl.len().alias("skus"),
                  pl.col("total_units").sum().alias("units"),
                  pl.col("oos_rate").mean().round(3).alias("oos_rate"))
             .with_columns((100 * pl.col("units") / pl.col("units").sum()).round(1).alias("units_pct"))
             .sort("units", descending=True))
display(summary)

fig, ax = plt.subplots(figsize=(7, 5))
sns.scatterplot(data=cls.to_pandas(), x="adi", y="cv2", hue="demand_class",
                size="total_units", sizes=(10, 300), alpha=0.6, ax=ax)
ax.axvline(1.32, ls="--", c="grey"); ax.axhline(0.49, ls="--", c="grey")
ax.set(xscale="log", yscale="log", xlabel="ADI (avg interval between sales)",
       ylabel="CV2 of non-zero demand", title="demand classification")

## Revenue concentration

Always know how concentrated the book is: optimising average WAPE across the whole
catalogue optimises mostly for SKUs nobody cares about.

In [ ]:
by_sku = (panel.group_by(C.SKU)
               .agg(pl.col(C.TARGET).sum().alias("units"),
                    pl.col("revenue").sum().alias("revenue"))
               .sort("revenue", descending=True)
               .collect()
               .with_columns((pl.col("revenue").cum_sum() / pl.col("revenue").sum()).alias("cum_share")))
n80 = by_sku.filter(pl.col("cum_share") <= 0.8).height
print(f"{n80} SKUs ({n80/by_sku.height:.0%}) make 80% of revenue")
by_sku.head(15)

## Seasonality

Day-of-week and month-of-year shape, plus the retail event spikes.

In [ ]:
cal = (panel.group_by(C.DATE).agg(pl.col(C.TARGET).sum().alias("units")).sort(C.DATE).collect())

fig, ax = plt.subplots(2, 1, figsize=(13, 7))
ax[0].plot(cal.get_column(C.DATE), cal.get_column("units"), lw=0.6)
ax[0].set(title="total daily units, all SKUs", ylabel="units")
roll = cal.with_columns(pl.col("units").rolling_mean(28).alias("ma28"))
ax[1].plot(roll.get_column(C.DATE), roll.get_column("ma28"), lw=1.5, c="crimson")
ax[1].set(title="28-day moving average", ylabel="units")
plt.tight_layout()

In [ ]:
seas = (panel.with_columns(pl.col(C.DATE).dt.weekday().alias("dow"),
                           pl.col(C.DATE).dt.month().alias("month"))
             .group_by("dow").agg(pl.col(C.TARGET).mean().alias("units")).sort("dow").collect())
month = (panel.with_columns(pl.col(C.DATE).dt.month().alias("month"))
              .group_by("month").agg(pl.col(C.TARGET).mean().alias("units")).sort("month").collect())

fig, ax = plt.subplots(1, 2, figsize=(13, 4))
sns.barplot(x=seas.get_column("dow"), y=seas.get_column("units"), ax=ax[0])
ax[0].set(title="mean units by weekday (1=Mon)", xlabel="weekday")
sns.barplot(x=month.get_column("month"), y=month.get_column("units"), ax=ax[1])
ax[1].set(title="mean units by month", xlabel="month")
plt.tight_layout()

## Stock-out censoring

This is the most important plot in the notebook. Days flagged out of stock have
*suppressed* sales -- the observed zero is missing data, not zero demand. If you train
and score on them, you are teaching the model to predict your own stock-outs.

In [ ]:
oos = (panel.with_columns(pl.col(C.DATE).dt.year().alias("year"))
            .group_by("year")
            .agg(pl.col("is_oos").mean().alias("oos_rate"),
                 (pl.col("is_oos") & (pl.col(C.TARGET) == 0)).mean().alias("censored_zero_rate"),
                 pl.len().alias("rows"))
            .sort("year").collect())
display(oos)

fig, ax = plt.subplots(figsize=(11, 4))
ax.bar(oos.get_column("year"), oos.get_column("oos_rate"), label="any OOS")
ax.bar(oos.get_column("year"), oos.get_column("censored_zero_rate"), label="OOS and zero sales")
ax.set(title="share of SKU-days flagged out of stock", ylabel="share"); ax.legend()

## A few individual series

Nothing substitutes for looking at the actual lines.

In [ ]:
top = by_sku.head(6).get_column(C.SKU).to_list()
fig, axes = plt.subplots(3, 2, figsize=(14, 9), sharex=True)
for sku, a in zip(top, axes.ravel()):
    d = panel.filter(pl.col(C.SKU) == sku).sort(C.DATE).collect()
    a.plot(d.get_column(C.DATE), d.get_column(C.TARGET), lw=0.5, alpha=0.7)
    a.plot(d.get_column(C.DATE), d.get_column(C.TARGET).rolling_mean(28), lw=1.5, c="crimson")
    o = d.filter(pl.col("is_oos"))
    a.scatter(o.get_column(C.DATE), [0]*len(o), s=2, c="orange", label="OOS")
    a.set_title(sku, fontsize=10)
plt.tight_layout()